In [ ]:
#!pip install ProcessOptimizer

In [ ]:
import numpy as np
import ProcessOptimizer as po
from ProcessOptimizer.learning.gaussian_process.gpr import GaussianProcessRegressor
from ProcessOptimizer.learning.gaussian_process.kernels import (
    ConstantKernel, Matern, WhiteKernel
)

# Simple Booth function (no noise for clarity)
def Booth(x0, x1):
    return (x0 + 2*x1 - 7)**2 + (2*x0 + x1 - 5)**2

# 1) define your 2-D space
SPACE = po.Space([[0.0, 5.0], [0.0, 5.0]])

# 2) build an ARD Matern (two independent lengthscales) + nugget
kernel = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
    * Matern(
        length_scale=[1.0, 1.0],            # ← ARD! one ℓ per input
        length_scale_bounds=(1e-2, 1e2),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e1)
    )
)

# 3) wrap it in PO’s GPR _with_ restarts
gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=5,
    random_state=0
)

# 4) pass that into the optimizer
opt = po.Optimizer(
    SPACE,
    base_estimator=gpr,
    n_initial_points=2
)

# 5) run a few iterations and print the ARD length-scales
for i in range(50):
    x = opt.ask()
    y = Booth(*x)
    opt.tell(x, y)

    if opt.models:
        # the last fitted GP is in opt.models[-1]
        gp_model = opt.models[-1]
        # for ConstantKernel * Matern + WhiteKernel, the Matern is at k1.k2
        ls = gp_model.kernel_.k1.k2.length_scale   # [ℓ₀, ℓ₁]
        print(f"Iter {i:2d}: length_scales = {ls}")
    else:
        print(f"Iter {i:2d}: No model fitted yet.")

In [ ]:
import numpy as np
from skopt import Optimizer
from skopt.space import Real
from skopt.learning import GaussianProcessRegressor
from skopt.learning.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

# Simple Booth function (no noise for clarity)
def Booth(x0, x1):
    return (x0 + 2*x1 - 7)**2 + (2*x0 + x1 - 5)**2

# 1) define your 2-D search space
dimensions = [
    Real(0.0, 5.0, name="x0"),
    Real(0.0, 5.0, name="x1"),
]

# 2) build an ARD Matern (two independent length-scales) + nugget
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=[1.0, 1.0],            # ← ARD: one ℓ per input
        length_scale_bounds=(1e-2, 1e2),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e1)
    )
)

# 3) wrap it in scikit-optimize’s GPR with restarts
gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=5,
    random_state=0
)

# 4) pass that into the scikit-optimize Optimizer
opt = Optimizer(
    dimensions=dimensions,
    base_estimator=gpr,
    acq_func="EI",         # or "PI", "LCB"
    random_state=0
)

# 5) run a few iterations and print the ARD length-scales
for i in range(20):
    x = opt.ask()
    y = Booth(*x)
    opt.tell(x, y)

    if opt.models:
        # the last fitted GP is in opt.models[-1]
        gp_model = opt.models[-1]
        # for ConstantKernel * Matern + WhiteKernel, the Matern is at k1.k2
        ls = gp_model.kernel_.k1.k2.length_scale   # [ℓ₀, ℓ₁]
        print(f"Iter {i:2d}: length_scales = {ls}")
    else:
        print(f"Iter {i:2d}: No model fitted yet.")

In [ ]:
import numpy as np
import ProcessOptimizer as po

def Booth(x0, x1):
    booth = (x0 + 2 * x1 - 7)**2 + (2 * x0 + x1 - 5)**2
    noise = 1 + 0.05 * (2 * np.random.rand() - 1)
    return (booth * noise)

SPACE = po.Space([[0.0, 5.0], [0.0, 5.0]])
opt = po.Optimizer(SPACE,
                   base_estimator = "GP",
                   n_initial_points = 2)

for i in range(20):
    x = opt.ask()
    y = Booth(*x)
    opt.tell(x, y)

    if opt.models:
        # the last fitted GP is in opt.models[-1]
        gp_model = opt.models[-1]
        # for ConstantKernel * Matern + WhiteKernel, the Matern is at k1.k2
        ls = gp_model.kernel_.k1.k2.length_scale   # [ℓ₀, ℓ₁]
        print(f"Iter {i:2d}: length_scales = {ls}")
    else:
        print(f"Iter {i:2d}: No model fitted yet.")

result = opt.get_result()
po.plot_objective(result)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import pandas as pd


# For reproducibility
np.random.seed(42)

In [ ]:
# Define benchmark functions.

eps = 1e-6

def aniso_sphere(x, y):
    a = 2
    b = 1
    z = (a*x)**2 + (b*y)**2 + x*y + x
    return 1/(z+1)


def booth(x, y):
    a = 2
    b = 7
    c = 2
    d = 5

    z = (x + a * y - b)**2 + (c * x + y - d)**2
    z_scaled = np.log10(z/100+1) # Avoid log(0) or log(negative)
    return 1/(z_scaled+1)

def three_hump_camel(x, y):
    a = 2
    b = 1.05
    c = 6

    z = a * x**2 - b * x**4 + (x**6)/c + x*y + y**2
    z_scaled = np.log10(z/100+1) # Avoid log(0) or log(negative)
    return 1/(z+1)

def rosenbrock(x, y):
    a = 1
    b = 100
    z = (a - x)**2 + b * (y - x**2)**2

    z_scaled = np.log10(z/100+1) # Avoid log(0) or log(negative)
    return 1/(z_scaled+1)

def himmelblau(x, y):
    a = 11
    b = 7
    z= (x**2 + y - a)**2 + (x + y**2 - b)**2

    z_scaled = np.log10(z/100+1) # Avoid log(0) or log(negative)    
    return 1/(z_scaled+1)

def hosaki(x, y):
    a = 8
    b = 7 
    c = 7/3
    d = 1/4 

    z = (1 - a * x + b * x**2 - c * x**3 + d * x**4) * (y**2) * np.exp(-y)
    z_min = 2.345811576101292 # Minimum value of the function

    return 1/(z+z_min+1)

# Create a dictionary that maps each benchmark name to a configuration that includes:
# - The benchmark function.
# - The x limits (domain for x).
# - The y limits (domain for y).
benchmark_config = {
    'aniso_sphere': {
        'func': aniso_sphere,
        'xlim': (-1, 1),
        'ylim': (-1, 1)
    },
    'booth': {
        'func': booth,
        'xlim': (-10, 10),
        'ylim': (-10, 10)
    },
    'three_hump_camel': {
        'func': three_hump_camel,
        'xlim': (-2, 2),
        'ylim': (-2, 2)
    },
    'rosenbrock': {
        'func': rosenbrock,
        'xlim': (-2, 2),
        'ylim': (-1, 3)
    },
    'himmelblau': {
        'func': himmelblau,
        'xlim': (-6, 6),
        'ylim': (-6, 6)
    },
    'hosaki': {
        'func': hosaki,
        'xlim': (0, 5),
        'ylim': (0, 5)
    }   
}

In [ ]:
def function_test(x, y, xvar_noise=0, z_noise_factor=0, z_noise_type='normal', benchmark_name='aniso_sphere'):
    """
    Evaluate a 2D benchmark function with optional noise.

    This function computes the value of a selected benchmark function 
    and applies optional noise to simulate variability in the output.
        x (float or np.ndarray): The x-coordinate(s) of the input point(s).
        y (float or np.ndarray): The y-coordinate(s) of the input point(s).
        xvar_noise (float, optional): Placeholder for an unused noise variable. Default is 0.
        z_noise_factor (float, optional): Scaling factor for the noise applied to the function value. Default is 0.
        z_noise_type (str, optional): Type of noise to apply (lognormal, normal or uniform). Default is 'normal'.
        benchmark_name (str, optional): The name of the benchmark function to evaluate. 
                                        Must be a key in the `benchmark_config` dictionary. Default is 'aniso_sphere'.
        np.ndarray: The computed function value(s) with added noise.
    
    Raises:
        ValueError: If the specified `benchmark_name` is not found in the `benchmark_config` dictionary.
    
    Notes:
        - The function uses a combination of Gaussian and uniform noise to generate variability.
        - The noise is scaled by the `z_noise_factor` parameter and applied multiplicatively to the function value.
        - The `benchmark_config` dictionary must be defined globally and contain the benchmark functions and their configurations.
    """
    
    # Normalize inputs to for larger standard Rosenbrock function
    x = x 
    y = y

    _ = xvar_noise # Not used

    
    # Create a dictionary to map text to function.
    # This allows us to easily switch between different benchmark functions.


    config = benchmark_config.get(benchmark_name)

    if config is None:
        raise ValueError(f"Benchmark '{benchmark_name}' is not configured.")

    # Compute the function value using the selected benchmark function.
    f_val = config['func'](x, y)

    # Noise parameters
    noise_std = 1.0
    noise_range = 1

    # Apply noise to the function value
  
    if z_noise_type == 'beta':
        # Generate beta noise

        # mean = 0.8 and std = 0.1
        # https://www.desmos.com/calculator/kx83qio7yl
        z_noise = np.random.beta(a=12.8, b=3.2, size=f_val.shape)

    elif z_noise_type == 'normal':
        # Generate Gaussian noise
        z_noise = np.random.normal(loc=0.0,
                                    scale=noise_std,
                                    size=f_val.shape)
    elif z_noise_type == 'uniform':
        # Generate uniform noise
        z_noise  = np.random.uniform(low=-noise_range/2,
                                   high= noise_range/2,
                                   size=f_val.shape)
    else:
        raise ValueError(f"Unknown noise type: {z_noise_type}")


    # Apply multiplicative noise to the function value
    z = f_val * (1.0 + z_noise_factor * z_noise)

    return z

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Select benchmark and noise level
selected_benchmark = 'three_hump_camel'      # or 'himmelblau'
z_noise_factor    = 1        # relative noise amplitude

# Pull in your benchmark config & function
config = benchmark_config[selected_benchmark]
func   = config['func']

# Build grid
x_vals = np.linspace(*config['xlim'], 400)
y_vals = np.linspace(*config['ylim'], 400)
X, Y   = np.meshgrid(x_vals, y_vals)

# Compute clean function on the whole grid at once
F = func(X, Y)

# Noise parameters
noise_std   = 1.0

# Generate per‐point noise arrays
gauss_noise= np.random.normal(loc=0.0, scale=noise_std, size=F.shape)

beta_noise = np.random.beta(a=12.8, b=3.2, size=F.shape)

# Combine and apply multiplicative noise
z_noise =  0*gauss_noise + beta_noise # zero‐mean noise

Z_noisy     = F * (1 + z_noise_factor * z_noise)

# Plot
plt.figure(figsize=(12, 10))
contour_noisy = plt.contourf(X, Y, Z_noisy, levels=100, cmap='viridis')
plt.colorbar(contour_noisy, label='z')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Contour Plot of Yield over x and y for {selected_benchmark.capitalize()} (With Noise)')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt



# List of benchmarks to plot
benchmarks = list(benchmark_config.keys())

# Noise and grid settings
z_noise_factor = 0.1  # relative noise amplitude

resolution = 200  # grid resolution per axis

# Grid layout
rows, cols = 3, 2
fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))

for idx, name in enumerate(benchmarks):
    conf = benchmark_config[name]
    func = conf['func']
    
    # Build meshgrid
    x_vals = np.linspace(*conf['xlim'], resolution)
    y_vals = np.linspace(*conf['ylim'], resolution)
    X, Y = np.meshgrid(x_vals, y_vals)
    
    # Compute clean function
    F = func(X, Y)
    
    # Noise parameters
    noise_std   = 1.0

    # Generate per‐point noise arrays
    gauss_noise= np.random.normal(loc=0.0, scale=noise_std, size=F.shape)

    beta_noise = np.random.beta(a=12.8, b=3.2, size=F.shape)

    # Combine and apply multiplicative noise
    z_noise =  gauss_noise 

    Z_noisy     = F * (1 + z_noise_factor * z_noise)
    
    # Plot
    ax = axes[idx // cols, idx % cols]
    cs = ax.contourf(X, Y, Z_noisy, levels=100, cmap='viridis')
    ax.set_title(name.capitalize())
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    fig.colorbar(cs, ax=ax)

# Remove any unused subplots
for idx in range(len(benchmarks), rows * cols):
    fig.delaxes(axes[idx // cols, idx % cols])

plt.tight_layout()
plt.show()

In [ ]:
import ProcessOptimizer as po
from ProcessOptimizer.learning.gaussian_process.gpr import GaussianProcessRegressor
from ProcessOptimizer.learning.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel


# -------------------------
# Set Up the Optimization Using ProcessOptimizer
# -------------------------
selected_benchmark = 'aniso_sphere'
z_noise_factor = 0.0
initial_length_scales = [1.0, 1.0, 1.0]  # Initial length scales for the kernel
list_length_scale_bounds = [(1e-1, 1e1), (1e-3, 1e3)]
length_scale_bounds = list_length_scale_bounds[1]

xi = 0.1  # Exploration-exploitation trade-off parameter

config = benchmark_config.get(selected_benchmark)
if config is None:
    raise ValueError(f"Benchmark '{selected_benchmark}' is not configured.")

space = po.Space([
    po.Real(config['xlim'][0], config['xlim'][1], name='x1'),
    po.Real(config['ylim'][0], config['ylim'][1], name='x2'),
    po.Real(-3, 3, name='syn_noise')  # The third 'fake' dimension
])

# Build an ARD Matern kernel with bounds + a little white noise
# ARD: Automatic Relevance Determination
kernel = (
    ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
    * Matern(
        length_scale=initial_length_scales,          # ← ARD: one lengthscale per dim. Corrected 0.0 to 1.0 for the 3rd dim.
        length_scale_bounds=length_scale_bounds,       # Bounds for all length scales
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,                      # Initial noise level
        noise_level_bounds=(1e-10, 1e1)        # Bounds for noise level
    )
)

# Create your own GPR with restarts
gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,    # allow hyperparameter restarts (optimization)
    random_state=0
)

# Pass your custom GPR into ProcessOptimizer
opt = po.Optimizer(
    space,
    base_estimator=gpr,         # Use your configured GPR
    acq_func="PI",
    random_state=0,              # For reproducibility of Optimizer's actions
    n_initial_points=30,        # Number of initial random points (Def. Screening Design 2n+1
    lhs=True # Latin Hypercube Sampling
)

# The following line for acq_func_kwargs is generally fine for 'xi',
opt.acq_func_kwargs = {"xi": xi }


# --- This part seems like a single initial step, let's integrate it into the loop logic ---
# x1_new, x2_new, x3_new = opt.ask()
# y_new = -function_test(x1_new, x2_new, x3_new, z_noise_factor=z_noise_factor, benchmark_name=selected_benchmark)
# opt.tell([x1_new, x2_new, x3_new], y_new)
# ---

results = []
n_experiments = 50 # Number of optimization iterations (after any initial points)

print("Starting optimization loop...")
for i in range(n_experiments):
    x_suggested = opt.ask() # Get suggestion from optimizer
    # Evaluate the candidate using function_test. Ensure x_suggested is unpacked correctly for 3 dims.
    y_observed = -function_test(x_suggested[0], x_suggested[1], x_suggested[2],
                                z_noise_factor=z_noise_factor, benchmark_name=selected_benchmark)

    # Update the optimizer with the observation
    model_result = opt.tell(x_suggested, y_observed)

    if opt.models:
        gp_model = opt.models[-1] # Get the last fitted GPR model
        # Access kernel parameters: ConstantKernel * Matern + WhiteKernel
        # gp_model.kernel_ is the fitted kernel
        # gp_model.kernel_.k1 is ConstantKernel * Matern
        # gp_model.kernel_.k1.k2 is Matern
        try:
            current_length_scales = gp_model.kernel_.k1.k2.length_scale
        except AttributeError:
            print(f"Iter {i:2d}: Could not extract length scales from kernel structure: {gp_model.kernel_}")
    else:
        print(f"Iter {i:2d}: No model fitted yet.") # Should not happen after first opt.tell()
        current_length_scales = initial_length_scales

    # Append iteration, inputs, objective, and length‐scales
    # Ensure correct indexing for length_scales if its structure is certain
    res_data = [i, x_suggested[0], x_suggested[1], x_suggested[2], -y_observed, *current_length_scales]
    results.append(res_data)

    if (i + 1) % 10 == 0: # Print progress
        print(f"Iter {i+1:3d}: Length Scales = {current_length_scales}")


# Convert the results list to a pandas DataFrame
column_names = ['Iteration', 'x1', 'x2', 'x3_fake_variable', 'y_observed', 'x1_length_scale', 'x2_length_scale', 'x3_fake_length_scale']
results_df = pd.DataFrame(results, columns=column_names)

# Set the index to the iteration number
results_df.set_index('Iteration', inplace=True)

print("\nResults DataFrame:")
print(results_df.head())
print("...")
print(results_df.tail())

https://scikit-optimize.github.io/stable/modules/generated/skopt.gp_minimize.html

In [ ]:
results_df.plot(subplots=True, figsize=(12, 8))

In [ ]:
po.plot_objective(result=model_result, pars="expected_minimum", show_confidence=True)

In [ ]:
po.plot_convergence(model_result);


In [ ]:
po.plots.plot_regret(model_result);


Multiple scenerios simulation

In [ ]:
import numpy as np
import pandas as pd
import ProcessOptimizer as po
from joblib import Parallel, delayed

# --- Parameters ---
n_simulations = 3
n_experiments = 100

n_initial_runs = 9  # Number of initial random experiments (Def. Screening Design 2n+1)

xis = [0.0, 0.1, 10]  # xi values for the acquisition function.
initial_length_scales = [1.0, 1.0, 1.0]  # Initial length scales for the kernel

list_length_scale_bounds = [(1e-1, 1e1), (1e-3, 1e3)]
list_acq_func = ["LCB", "PI", "EI", "gp_hedge"]

selected_benchmarks = benchmark_config.keys()  # All benchmarks available in the config.

z_noise_factors = [0, 0.0001, 0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1]



# --- The run_simulation function ---
def run_simulation(n_sim):
    """
    Run one simulation (with all benchmarks and inner parameter loops)
    and return a list of result rows.
    """
    sim_results = []
    print(f"Simulation {n_sim+1}/{n_simulations}")
    
    # Generate random seed for this simulation.
    random_seed = np.random.randint(0, 10000)
    
    # Iterate over each benchmark.
    for selected_benchmark in selected_benchmarks:
        print(f"  Benchmark: {selected_benchmark}")
        # Get benchmark configuration (must be defined elsewhere).
        config = benchmark_config.get(selected_benchmark)
        if config is None:
            raise ValueError(f"Benchmark '{selected_benchmark}' is not configured.")
    
        # Define the search space based on the benchmark's x and y limits.
        space = po.Space([
            po.Real(config['xlim'][0], config['xlim'][1], name='x1'),
            po.Real(config['ylim'][0], config['ylim'][1], name='x2'),
            po.Real(-3, 3, name='syn_noise')
        ])
    
        # Loop over different noise factors.
        for z_noise_factor in z_noise_factors:
            for acq_func_selected in list_acq_func:
                for len_scale_bounds in list_length_scale_bounds:
                    for xi_value in xis:

                        # Build an ARD Matern kernel with bounds + a little white noise
                        # ARD: Automatic Relevance Determination
                        kernel = (
                            ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3))
                            * Matern(
                                length_scale=initial_length_scales,          # ← ARD: one lengthscale per dim. Corrected 0.0 to 1.0 for the 3rd dim.
                                length_scale_bounds=len_scale_bounds,       # Bounds for all length scales
                                nu=2.5
                            )
                            + WhiteKernel(
                                noise_level=1e-6,                      # Initial noise level, TODO modify?
                                noise_level_bounds=(1e-10, 1e1)        # Bounds for noise level
                            )
                        )

                        # Create your own GPR with restarts
                        gpr = GaussianProcessRegressor(
                            kernel=kernel,
                            normalize_y=True,
                            n_restarts_optimizer=10,    # allow hyperparameter restarts (optimization)
                            random_state=random_seed  # TODO modifify?
                        )

                        # Initialize the optimizer with the current settings.
                        opt = po.Optimizer(space, base_estimator=gpr,
                                           acq_func=acq_func_selected,
                                            random_state=random_seed,              # For reproducibility of Optimizer's actions
                                            n_initial_points=n_initial_runs,        # Number of initial random points 
                                            lhs=True # Latin Hypercube Sampling
                        )

                        opt.acq_func_kwargs = {"xi": xi_value}
    
                        # Run experiments using the optimizer.
                        for i in range(n_experiments):
                            x1_new, x2_new, x3_new = opt.ask()
    
                            # Evaluate the negative yield (assuming minimization) using function_test.
                            # x3_new is passed as syn_noise; the current noise factor and benchmark name are used.
                            y_new = -function_test(x1_new, x2_new, x3_new,
                                                   z_noise_factor=z_noise_factor,
                                                   benchmark_name=selected_benchmark)
    
                            # Feed the evaluated value back to the optimizer.
                            opt.tell([x1_new, x2_new, x3_new], y_new)

                            if opt.models:
                                gp_model = opt.models[-1] # Get the last fitted GPR model
                                try:
                                    current_length_scales = gp_model.kernel_.k1.k2.length_scale
                                except AttributeError:
                                    print(f"Iter {i:2d}: Could not extract length scales from kernel structure: {gp_model.kernel_}")
                            else:
                                # print(f"Iter {i:2d}: No model fitted yet.") 
                                current_length_scales = initial_length_scales
    
                            # Append parameters and settings to the results.
                            sim_results.append([
                                n_sim + 1, i + 1, x1_new, x2_new, x3_new, -y_new, xi_value,
                                *len_scale_bounds,
                                *current_length_scales,
                                acq_func_selected,
                                z_noise_factor, selected_benchmark
                            ])
    return sim_results

# --- Run simulations in parallel ---
# Each simulation (0 to n_simulations-1) is run concurrently.
results_list = Parallel(n_jobs=-1, verbose=10)(
    delayed(run_simulation)(sim) for sim in range(n_simulations)
)

# Flatten the list of simulation results.
results = [row for sim_results in results_list for row in sim_results]

# --- Convert results to a DataFrame and save ---
columns = ['n_sim', 'run', 'x1', 'x2', 'x3_fake_var', 'y_new', 'xi', 
           'low_length_scale_bound', 'high_length_scale_bound',
            'x1_length_scale', 'x2_length_scale', 'x3_fake_var_length_scale',
            'acq_func', 
           'z_noise_factor', 'selected_benchmark']
results_df = pd.DataFrame(results, columns=columns)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_DIR / "results.csv", index=False)
print(results_df)